# LoRA safetensors → Draw Things `.ckpt` (Colab, minimal build)

Converts a LoRA `.safetensors` file to the Draw Things `.ckpt` format with Draw Things' own importer
(`ModelOp.LoRAImporter`), built as a **small standalone Swift tool** instead of the full `media-generation-kit-cli`.
Only the targets the importer needs are compiled (ModelOp, Diffusion, ModelZoo, DataModels and the NNC/ccv core).
The image generators, gRPC servers, Vision/CoreML code and the CLI itself are never built.

1. **Runtime → Run all.** Cell 2 installs Swift. Cell 3 builds the tool (expect roughly 10–30 minutes the first time).
2. Cell 4 asks you to pick your `.safetensors` file and downloads the `.ckpt`.

**Status: experimental, not yet run on Colab.** If cell 3 fails, its last lines print the compiler errors; copy them
into the dtFlow issue. Use the free CPU runtime: a GPU gives no speed-up.
Your LoRA is processed inside your own Colab session; nothing is sent anywhere else.

In [ ]:
# @title 1. Choose the Draw Things source and install the Swift toolchain
import hashlib
import json
import os
import platform
import shlex
import shutil
import subprocess
import tempfile
import time
import urllib.request
from pathlib import Path

# @markdown Draw Things source: "newer" knows the newest model families (Krea 2, FLUX.2 9B/4B, FLUX.3, LTX-2, ...)
# @markdown and needs Swift 6.1. "older" is the first proven build (Swift 6.0.3), without those families.
draw_things_version = "newer"  # @param ["newer", "older"]
SWIFT_VERSION = "6.1.2" if draw_things_version == "newer" else "6.0.3"
SWIFT_ROOT = "/opt/swift"


def sh(command, cwd=None, every=1, log=None):
    """Run a shell command, stream its output, optionally keep it in a log file."""
    print("$", command, flush=True)
    process = subprocess.Popen(
        command,
        shell=True,
        executable="/bin/bash",
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        bufsize=1,
    )
    count = 0
    handle = open(log, "w") if log else None
    try:
        for line in process.stdout:
            count += 1
            if handle:
                handle.write(line)
            if count % every == 0:
                print(line.rstrip(), flush=True)
    finally:
        if handle:
            handle.close()
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {command}")


def show_progress(blocks, block_size, total):
    """Print download progress about every 5 percent."""
    if total > 0:
        percent = min(100, blocks * block_size * 100 // total)
        if percent // 5 != show_progress.last:
            show_progress.last = percent // 5
            print(f"  downloaded {percent}%", flush=True)


show_progress.last = -1

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("This notebook needs the x86_64 Linux Colab runtime.")

release = {}
with open("/etc/os-release") as handle:
    for line in handle:
        if "=" in line:
            key, value = line.rstrip().split("=", 1)
            release[key] = value.strip('"')
ubuntu = release.get("VERSION_ID", "")
if ubuntu not in ("22.04", "24.04"):
    raise RuntimeError(f"Unsupported Ubuntu {ubuntu!r}; this notebook supports 22.04 and 24.04.")

# The Swift toolchain tarball already bundles clang, libdispatch and Foundation, so the system clang/LLVM
# (a large download that can time out) and OpenMP are not installed. Retry because the mirror is sometimes flaky.
APT_PACKAGES = (
    "git binutils libc6-dev libcurl4-openssl-dev libedit2 libncurses-dev libsqlite3-dev libxml2-dev "
    "zlib1g-dev libpng-dev libjpeg-dev libblas-dev liblapack-dev libssl-dev uuid-dev pkg-config tzdata unzip"
)
for attempt in range(1, 4):
    try:
        sh(
            "apt-get update -o Acquire::Retries=5 && DEBIAN_FRONTEND=noninteractive "
            f"apt-get install -y --fix-missing -o Acquire::Retries=5 {APT_PACKAGES}"
        )
        break
    except RuntimeError:
        if attempt == 3:
            raise
        print(f"apt failed (attempt {attempt}); retrying in 20 seconds", flush=True)
        time.sleep(20)

folder = f"swift-{SWIFT_VERSION}-RELEASE-ubuntu{ubuntu}"
if not os.path.exists(f"{SWIFT_ROOT}/{folder}/usr/bin/swift"):
    url = (
        f"https://download.swift.org/swift-{SWIFT_VERSION}-release/ubuntu{ubuntu.replace('.', '')}/"
        f"swift-{SWIFT_VERSION}-RELEASE/{folder}.tar.gz"
    )
    os.makedirs(SWIFT_ROOT, exist_ok=True)
    archive = f"/tmp/{folder}.tar.gz"
    print("downloading", url, flush=True)
    urllib.request.urlretrieve(url, archive, show_progress)
    print("extracting (a progress line every 2000 files)", flush=True)
    sh(f"tar -xzvf {archive} -C {SWIFT_ROOT}", every=2000)
    os.remove(archive)

SWIFT_BIN = f"{SWIFT_ROOT}/{folder}/usr/bin"
if SWIFT_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = SWIFT_BIN + os.pathsep + os.environ["PATH"]
sh("swift --version")

In [ ]:
# @title 2. Build the minimal converter (about 10–30 min the first time)
import re

# @markdown Optional: keep the built tool in Google Drive so later sessions skip the build.
use_drive_cache = True  # @param {type:"boolean"}
# @markdown Extra `swift build` flags, only for experiments (leave empty normally).
extra_build_flags = ""  # @param {type:"string"}

REPO_URL = "https://github.com/drawthingsai/draw-things-community.git"
REVISIONS = {
    "newer": "db37e657ad329dd16c2f1200bb47f940369cd9f4",
    "older": "d473a2f148b3e7dc9b90d0b7cfccc5cda999eb66",
}
PINNED_REVISION = REVISIONS[draw_things_version]
TOOL_NAME = "lora-convert"
WORK = Path("/content/dt-build")
SOURCE = WORK / "draw-things-community"
BINARY = SOURCE / ".build" / "release" / TOOL_NAME
BUILD_LOG = WORK / "build.log"
SWIFT_MARKER = WORK / "swift-version.txt"
DRIVE_CACHE = Path("/content/drive/MyDrive/dtflow-lora-converter")

MAIN_SWIFT = r"""import Diffusion
import Foundation
import ModelOp
import ModelZoo

// Minimal Draw Things LoRA importer: .safetensors -> Draw Things .ckpt.
// Usage: lora-convert --input <file.safetensors> --output <file.ckpt> [--scale 1.0]

func fail(_ message: String) -> Never {
  FileHandle.standardError.write(Data(("error: " + message + "\n").utf8))
  exit(1)
}

var input: String? = nil
var output: String? = nil
var scale = 1.0
var versionName: String? = nil
var arguments = Array(CommandLine.arguments.dropFirst())
while !arguments.isEmpty {
  let flag = arguments.removeFirst()
  guard !arguments.isEmpty else { fail("missing value for \(flag)") }
  let value = arguments.removeFirst()
  switch flag {
  case "--input": input = value
  case "--output": output = value
  case "--version": versionName = value
  case "--scale":
    guard let parsed = Double(value) else { fail("invalid --scale \(value)") }
    scale = parsed
  default: fail("unknown option \(flag)")
  }
}
guard let input = input, let output = output else {
  fail("usage: lora-convert --input <file.safetensors> --output <file.ckpt> [--scale 1.0] [--version <family>]")
}

let sourceFile = URL(fileURLWithPath: input).standardizedFileURL
let destinationFile = URL(fileURLWithPath: output).standardizedFileURL
guard FileManager.default.fileExists(atPath: sourceFile.path) else {
  fail("input not found: \(sourceFile.path)")
}

let tempDirectory = FileManager.default.temporaryDirectory.appendingPathComponent(
  "dtflow-lora-\(UUID().uuidString)", isDirectory: true)
try FileManager.default.createDirectory(at: tempDirectory, withIntermediateDirectories: true)
defer { try? FileManager.default.removeItem(at: tempDirectory) }

let outputName = destinationFile.deletingPathExtension().lastPathComponent
let outputFilename = destinationFile.lastPathComponent
ModelZoo.externalUrls = [tempDirectory]

var forcedVersion: ModelVersion? = nil
if let name = versionName {
  switch name {
  case "v1": forcedVersion = .v1
  case "v2": forcedVersion = .v2
  case "sdxlBase": forcedVersion = .sdxlBase
  case "sdxlRefiner": forcedVersion = .sdxlRefiner
  case "ssd1b": forcedVersion = .ssd1b
  case "sd3": forcedVersion = .sd3
  case "sd3Large": forcedVersion = .sd3Large
  case "pixart": forcedVersion = .pixart
  case "flux1": forcedVersion = .flux1
  case "hunyuanVideo": forcedVersion = .hunyuanVideo
  case "wan21_1_3b": forcedVersion = .wan21_1_3b
  case "wan22_5b": forcedVersion = .wan22_5b
  case "wan21_14b": forcedVersion = .wan21_14b
  case "hiDreamI1": forcedVersion = .hiDreamI1
  case "qwenImage": forcedVersion = .qwenImage
  case "zImage": forcedVersion = .zImage
  case "flux2": forcedVersion = .flux2
  // __EXTRA_VERSIONS__
  default: fail("unknown --version \(name); the newest families need draw_things_version = newer in cell 1")
  }
}

var lastPercent = -1
do {
  let result = try ModelOp.LoRAImporter.import(
    downloadedFile: sourceFile.path,
    name: outputName,
    filename: outputFilename,
    scaleFactor: scale,
    forceVersion: forcedVersion,
    progress: { progress in
      let percent = Int(Double(progress) * 100)
      if percent / 10 != lastPercent / 10 {
        lastPercent = percent
        print("  converting \(percent)%")
        fflush(stdout)
      }
    })
  print("model version: \(result.0)")
} catch ModelOp.LoRAImporter.Error.modelVersionFailed {
  fail("could not detect the model version of this LoRA; pass --version <family> (for example flux1)")
} catch {
  fail("conversion failed: \(error)")
}

try FileManager.default.createDirectory(
  at: destinationFile.deletingLastPathComponent(), withIntermediateDirectories: true)
if FileManager.default.fileExists(atPath: destinationFile.path) {
  try FileManager.default.removeItem(at: destinationFile)
}
try FileManager.default.moveItem(
  at: tempDirectory.appendingPathComponent(outputFilename), to: destinationFile)
print("wrote \(destinationFile.path)")
"""

# Model families that only exist in the newer Draw Things source (the Swift cases do not compile on the older one).
NEWER_FAMILIES = [
    "krea2", "flux2_9b", "flux2_4b", "flux3_5b", "qwenImage2_1", "ernieImage", "mossImage",
    "minimaxH3", "ideogram4", "ltx2", "ltx2_3", "cosmos2_5_2b", "cosmos2_5_2_9b",
]  # fmt: skip
if draw_things_version == "newer":
    extra_cases = "".join(f'  case "{name}": forcedVersion = .{name}\n' for name in NEWER_FAMILIES)
else:
    extra_cases = ""
MAIN_SWIFT = MAIN_SWIFT.replace("  // __EXTRA_VERSIONS__\n", extra_cases)

# The cached binary is tied to this tool's source: changing the Swift code above invalidates it.
TOOL_STAMP = hashlib.sha256(MAIN_SWIFT.encode()).hexdigest()[:8]
STAMP_FILE = BINARY.with_name(TOOL_NAME + ".stamp")
CACHED_BINARY = DRIVE_CACHE / f"{TOOL_NAME}-{TOOL_STAMP}-{PINNED_REVISION[:7]}-swift{SWIFT_VERSION}-ubuntu{ubuntu}"

if use_drive_cache:
    from google.colab import drive

    drive.mount("/content/drive")

if use_drive_cache and CACHED_BINARY.is_file() and not BINARY.is_file():
    BINARY.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CACHED_BINARY, BINARY)
    BINARY.chmod(0o755)
    STAMP_FILE.write_text(TOOL_STAMP)
    print("restored the converter from Drive:", CACHED_BINARY)

if BINARY.is_file() and (not STAMP_FILE.is_file() or STAMP_FILE.read_text() != TOOL_STAMP):
    BINARY.unlink()
    print("the converter source changed since the last build; relinking")

if not BINARY.is_file():
    WORK.mkdir(parents=True, exist_ok=True)
    if not SOURCE.exists():
        sh(f"git clone --progress {REPO_URL} {SOURCE}")
    # Fetch the revision (it may be newer than the clone) and drop edits from a previous attempt or revision.
    if SWIFT_MARKER.is_file() and SWIFT_MARKER.read_text() != SWIFT_VERSION:
        shutil.rmtree(SOURCE / ".build", ignore_errors=True)
        print("the Swift version changed; cleared the old build folder")
    SWIFT_MARKER.write_text(SWIFT_VERSION)
    sh(f"git -C {SOURCE} fetch -q origin {PINNED_REVISION} || true")
    sh(f"git -C {SOURCE} reset -q --hard {PINNED_REVISION}")

    # Add one small executable target next to the library targets. Nothing else in the package is built.
    tool_dir = SOURCE / "Apps" / "LoraConvert"
    tool_dir.mkdir(parents=True, exist_ok=True)
    (tool_dir / "main.swift").write_text(MAIN_SWIFT)

    manifest = SOURCE / "Package.swift"
    text = manifest.read_text()
    if "LoraConvert" not in text:
        product = f'    .executable(name: "{TOOL_NAME}", targets: ["LoraConvert"]),\n'
        target = (
            "    .executableTarget(\n"
            '      name: "LoraConvert",\n'
            '      dependencies: ["ModelOp", "ModelZoo", "Diffusion", "DataModels"],\n'
            '      path: "Apps/LoraConvert"\n'
            "    ),\n"
        )
        products_anchor = "  products: [\n"
        targets_end = "\n  ]\n)"
        if text.count(products_anchor) != 1 or text.count(targets_end) != 1:
            raise RuntimeError("Package.swift layout changed upstream; the manifest edit needs updating.")
        text = text.replace(products_anchor, products_anchor + product, 1)
        text = text.replace(targets_end, "\n" + target + "  ]\n)", 1)
        manifest.write_text(text)
    print("added the", TOOL_NAME, "target to Package.swift")

    sh("swift package resolve", cwd=SOURCE, every=5)

    # ccv's SwiftPM manifest is written for Apple platforms: it defines HAVE_MPS / HAVE_ACCELERATE_FRAMEWORK /
    # USE_DISPATCH, links Metal frameworks and compiles the Metal sources. Rewrite it for a CPU-only Linux build.
    ccv_dir = SOURCE / ".build" / "checkouts" / "ccv"
    ccv_manifest = ccv_dir / "Package.swift"
    if not ccv_manifest.is_file():
        raise RuntimeError("ccv checkout not found; the dependency layout changed upstream.")
    manifest_text = ccv_manifest.read_text()
    if "dtflow-linux" not in manifest_text:
        drop = ('.define("HAVE_MPS")', '.define("HAVE_ACCELERATE_FRAMEWORK")', '.define("USE_DISPATCH")')
        out_lines = []
        for line in manifest_text.split("\n"):
            stripped = line.strip()
            if stripped.startswith(drop):
                continue
            if stripped.startswith('.linkedFramework("Accelerate")'):
                out_lines.append(line.replace('.linkedFramework("Accelerate")', '.linkedLibrary("blas")'))
                continue
            if stripped.startswith(".linkedFramework("):
                continue
            out_lines.append(line)
        manifest_text = "\n".join(out_lines)

        # Do not compile the Metal / Objective-C sources inside the NNC target.
        nnc_dir = ccv_dir / "lib" / "nnc"
        skip = {"mfa"}
        for path in nnc_dir.rglob("*"):
            relative = path.relative_to(nnc_dir)
            if relative.parts[0] in ("mps", "mfa"):
                continue
            if path.is_dir() and path.name == "mps":
                skip.add(relative.as_posix())
            elif path.is_file() and path.suffix in (".m", ".mm", ".metal"):
                skip.add(relative.as_posix())
        marker = '                "mps",\n'
        if manifest_text.count(marker) != 1:
            raise RuntimeError("ccv Package.swift layout changed; the manifest edit needs updating.")
        extra = "".join(f'                "{name}",\n' for name in sorted(skip))
        manifest_text = manifest_text.replace(marker, marker + extra, 1)

        # Replace the Metal compatibility target by an empty C target (the product name stays the same).
        start = manifest_text.index("        // MPS Compatibility layer for NNC")
        end = manifest_text.index("    ],\n    cLanguageStandard")
        stub = (
            "        .target(\n"
            '            name: "lib_nnc_mps_compat",\n'
            '            dependencies: ["C_swiftpm_nnc"],\n'
            '            path: "stubs/mps_compat",\n'
            '            publicHeadersPath: "include"\n'
            "        ),\n"
        )
        manifest_text = manifest_text[:start] + stub + manifest_text[end:] + "\n// dtflow-linux\n"
        stub_dir = ccv_dir / "stubs" / "mps_compat"
        (stub_dir / "include").mkdir(parents=True, exist_ok=True)
        (stub_dir / "include" / "lib_nnc_mps_compat.h").write_text("#pragma once\n")
        (stub_dir / "stub.c").write_text("int dtflow_mps_compat_unused;\n")
        ccv_manifest.write_text(manifest_text)
        print("ccv Package.swift rewritten for Linux; skipped Metal sources:", len(skip))

    # s4nnc's SwiftPM manifest has no SQLite on Linux: Store.swift does `import _SQLite3Shims` there (upstream's
    # Bazel build supplies it from nnc/C_sqlite3/shim.h). Add the same module as a system library.
    s4nnc_dir = SOURCE / ".build" / "checkouts" / "s4nnc"
    s4nnc_manifest = s4nnc_dir / "Package.swift"
    if not s4nnc_manifest.is_file():
        raise RuntimeError("s4nnc checkout not found; the dependency layout changed upstream.")
    s4nnc_text = s4nnc_manifest.read_text()
    if "dtflow-linux" not in s4nnc_text:
        zlib_target = "    // C_zlib - System zlib wrapper\n"
        nnc_deps = '        "C_zlib",\n      ],\n      path: "nnc",'
        if s4nnc_text.count(zlib_target) != 1 or s4nnc_text.count(nnc_deps) != 1:
            raise RuntimeError("s4nnc Package.swift layout changed; the manifest edit needs updating.")
        sqlite_target = '    .systemLibrary(\n      name: "_SQLite3Shims",\n      path: "nnc/C_sqlite3"\n    ),\n\n'
        s4nnc_text = s4nnc_text.replace(zlib_target, sqlite_target + zlib_target, 1)
        s4nnc_text = s4nnc_text.replace(
            nnc_deps, '        "C_zlib",\n        "_SQLite3Shims",\n      ],\n      path: "nnc",', 1
        )
        s4nnc_manifest.write_text(s4nnc_text + "\n// dtflow-linux\n")
        (s4nnc_dir / "nnc" / "C_sqlite3" / "module.modulemap").write_text(
            'module _SQLite3Shims [system] {\n  header "shim.h"\n  link "sqlite3"\n  export *\n}\n'
        )
        print("s4nnc: added the _SQLite3Shims system library")

    # On Linux glibc `fopen` returns an optional FILE pointer, which the Apple-oriented s4nnc code uses directly.
    # Unwrap it at the two exact places the compiler reported (not a blanket rewrite).
    file_fixes = {
        "DataFrameAddons.swift": [("    fclose(fileHandle)\n", "    fclose(fileHandle!)\n")],
        "Store.swift": [
            ('      let fp = fopen(externalStore, "rb")\n', '      let fp = fopen(externalStore, "rb")!\n')
        ],
    }
    for file_name, replacements in file_fixes.items():
        target_file = s4nnc_dir / "nnc" / file_name
        file_text = target_file.read_text()
        for before, after in replacements:
            if after in file_text:
                continue
            if file_text.count(before) != 1:
                print(f"warning: s4nnc {file_name}: expected line not found once, skipping: {before.strip()}")
                continue
            file_text = file_text.replace(before, after, 1)
            print(f"s4nnc {file_name}: unwrapped {before.strip()}")
        target_file.write_text(file_text)

    # Same optional-FILE problem in ModelOp's LoRAExporter (`let w = fopen(...)` used as non-optional).
    # Only plain `let/var x = fopen(...)` lines in ModelOp are unwrapped.
    fopen_line = re.compile(r"^(\s*(?:let|var)\s+\w+\s*=\s*fopen\(.*\))\s*$", re.MULTILINE)
    for swift_file in (SOURCE / "Libraries" / "ModelOp" / "Sources").rglob("*.swift"):
        file_text = swift_file.read_text()
        fixed_text, count = fopen_line.subn(r"\1!", file_text)
        if count:
            swift_file.write_text(fixed_text)
            print(f"ModelOp {swift_file.name}: unwrapped {count} fopen call(s)")

    # Pre-flight: list Apple-only imports in the sources the converter compiles, so likely failures show up now
    # instead of 20 minutes into the build. Imports inside #if canImport / #if os blocks are not listed.
    apple_import = re.compile(
        r"^\s*import\s+(CoreML|CoreGraphics|CoreImage|ImageIO|Vision|Metal\w*|Accelerate|CryptoKit|UIKit|AppKit|"
        r"Combine|OSLog|os|CFNetwork|Network|UniformTypeIdentifiers|AVFoundation|Darwin)\b"
    )
    closure_dirs = [
        "Libraries/DataModels",
        "Libraries/SwiftDiffusion/Sources",
        "Libraries/ModelZoo/Sources",
        "Libraries/ModelOp/Sources",
        "Libraries/Upscaler/Sources",
        "Libraries/WeightsCache/Sources",
        "Libraries/Tokenizer/Sources",
        "Libraries/LLM/Sources",
        "Libraries/Utils/Sources",
        "Vendors/ZIPFoundation/Sources",
    ]
    excluded_dirs = {"CoreML", "Preprocessors", "CoreMLModelManager", "UNetWrapper"}
    flagged = []
    for folder_name in closure_dirs:
        for swift_file in (SOURCE / folder_name).rglob("*.swift"):
            if excluded_dirs & set(swift_file.relative_to(SOURCE).parts):
                continue
            file_lines = swift_file.read_text(errors="replace").split("\n")
            for number, text_line in enumerate(file_lines):
                if apple_import.match(text_line):
                    context = "\n".join(file_lines[max(0, number - 4) : number])
                    if "#if" not in context:
                        flagged.append(f"{swift_file.relative_to(SOURCE)}:{number + 1}: {text_line.strip()}")
    print(f"pre-flight: {len(flagged)} unguarded Apple-only imports in the converter's sources")
    for entry in flagged[:30]:
        print("  ", entry)

    # Find cblas.h (its folder differs between the BLAS packages) and pass it to the C compiler.
    found = subprocess.run("find /usr/include -name cblas.h", shell=True, capture_output=True, text=True)
    cblas_dirs = sorted({str(Path(line).parent) for line in found.stdout.split() if line})
    print("cblas.h folders:", cblas_dirs)
    # ZIPFoundation does `import CZlib` on Linux but its manifest has no such module: provide a system modulemap
    # for the zlib headers (this is the part that cost the most time in the first notebook).
    czlib_dir = Path("/tmp/CZlib")
    czlib_dir.mkdir(exist_ok=True)
    (czlib_dir / "module.modulemap").write_text(
        'module CZlib [system] {\n  header "/usr/include/zlib.h"\n  link "z"\n  export *\n}\n'
    )
    czlib_flags = "-Xcc -I/usr/include -Xcc -I/tmp/CZlib -Xswiftc -I/tmp/CZlib"
    extra_build_flags = " ".join(
        [extra_build_flags, czlib_flags] + [f"-Xcc -I{folder}" for folder in cblas_dirs[:1]]
    ).strip()

    # Linux compile fix: ZIPFoundation's in-memory FILE pointer is optional on Linux (fopencookie),
    # but the vendored source passes it straight to fseeko. Unwrap it. Not fatal if upstream already changed it.
    memory_file = SOURCE / "Vendors/ZIPFoundation/Sources/ZIPFoundation/Archive+MemoryFile.swift"
    if memory_file.is_file():
        source_text = memory_file.read_text()
        before = "      if append {\n        fseeko(result, 0, SEEK_END)"
        after = "      if append, let result = result {\n        fseeko(result, 0, SEEK_END)"
        if source_text.count(before) == 1:
            memory_file.write_text(source_text.replace(before, after, 1))
            print("ZIPFoundation Linux patch applied")
        else:
            print("ZIPFoundation patch not needed or the source changed; skipping")

    jobs = os.cpu_count() or 2
    try:
        # Build the importer library first (ccv, s4nnc, Diffusion, ModelOp): if Linux support is missing,
        # it fails here after the dependency build instead of at the very end.
        sh(
            f"swift build -c release --target ModelOp -j {jobs} {extra_build_flags}",
            cwd=SOURCE,
            log=str(BUILD_LOG),
        )
        print("importer libraries built; linking the converter tool")
        sh(
            f"swift build -c release --product {TOOL_NAME} -j {jobs} {extra_build_flags}",
            cwd=SOURCE,
            log=str(BUILD_LOG),
        )
    except RuntimeError:
        print("\n===== BUILD FAILED. Compiler errors (copy these into the issue): =====")
        errors = [line.rstrip() for line in BUILD_LOG.read_text(errors="replace").splitlines() if "error:" in line]
        for line in errors[:40]:
            print(line)
        print(f"({len(errors)} error lines in total; the full log is {BUILD_LOG})")
        raise

    if use_drive_cache and BINARY.is_file():
        DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
        STAMP_FILE.write_text(TOOL_STAMP)
        shutil.copy2(BINARY, CACHED_BINARY)
        print("saved the converter to Drive:", CACHED_BINARY)

if not BINARY.is_file():
    raise RuntimeError("The build finished without producing the converter binary.")
STAMP_FILE.write_text(TOOL_STAMP)
print("converter ready:", BINARY)
print("draw-things-community revision:", PINNED_REVISION)

In [ ]:
# @title 3. Convert a LoRA (pick the file, then download the .ckpt)
# @markdown Scale is applied to the LoRA weights while converting. Leave 1.0 unless you know you need another value.
scale = 1.0  # @param {type:"number"}
# @markdown Model family of the LoRA. Leave on auto-detect first; if the converter says it could not detect the
# @markdown version (typical for LoRAs that do not train every block), pick the base model it was trained for.
model_family = "auto-detect"  # @param ["auto-detect", "krea2", "flux1", "flux2", "flux2_9b", "flux2_4b", "flux3_5b", "sd3", "sd3Large", "v1", "v2", "sdxlBase", "sdxlRefiner", "ssd1b", "pixart", "hunyuanVideo", "wan21_1_3b", "wan21_14b", "wan22_5b", "hiDreamI1", "qwenImage", "qwenImage2_1", "zImage", "ernieImage", "mossImage", "minimaxH3", "ideogram4", "ltx2", "ltx2_3", "cosmos2_5_2b", "cosmos2_5_2_9b"]
# @markdown Leave empty to pick a file from your computer. Or give a path inside Colab / Drive, e.g. /content/drive/MyDrive/my_lora.safetensors
source_path = ""  # @param {type:"string"}
# @markdown Optional, like the Draw Things app form. They are stored in the provenance JSON only (the `.ckpt`
# @markdown file itself has no name or trigger word; Draw Things keeps those in its own LoRA list).
lora_name = ""  # @param {type:"string"}
trigger_word = ""  # @param {type:"string"}
download_result = True  # @param {type:"boolean"}

if source_path.strip():
    source = Path(source_path.strip()).expanduser()
else:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Pick exactly one .safetensors file.")
    source = Path("/content") / next(iter(uploaded))

if not source.is_file():
    raise RuntimeError(f"Input not found: {source}")
if source.suffix.lower() != ".safetensors":
    print(f"warning: {source.name} does not end in .safetensors")


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


out_dir = Path("/content/lora_out")
out_dir.mkdir(exist_ok=True)
target = out_dir / f"{source.stem}_lora_f16.ckpt"
if target.exists():
    raise RuntimeError(f"{target} already exists; not overwriting it. Remove it first.")

with tempfile.TemporaryDirectory(prefix=".convert-", dir=out_dir) as staging:
    staged = Path(staging) / target.name
    command = [str(BINARY), "--input", str(source), "--output", str(staged)]
    if scale != 1.0:
        command += ["--scale", str(scale)]
    if model_family != "auto-detect":
        command += ["--version", model_family]
    sh(shlex.join(command))
    if not staged.is_file():
        raise RuntimeError("The converter finished without writing a .ckpt.")
    staged.replace(target)

size = target.stat().st_size
digest = sha256_of(target)
provenance = target.with_suffix(".json")
provenance.write_text(
    json.dumps(
        {
            "output": target.name,
            "size_bytes": size,
            "sha256": digest,
            "input": source.name,
            "input_sha256": sha256_of(source),
            "scale": scale,
            "model_family": model_family,
            "name": lora_name.strip() or source.stem,
            "trigger_word": trigger_word.strip(),
            "draw_things_community_revision": PINNED_REVISION,
            "swift_version": SWIFT_VERSION,
            "build": "minimal lora-convert target (ModelOp.LoRAImporter)",
        },
        indent=2,
    )
)
print("output:", target)
print("size:  ", size, "bytes")
print("sha256:", digest)

if download_result:
    from google.colab import files

    files.download(str(target))
    files.download(str(provenance))